# SPIES Practicum — Synthetic Data Replication

同构脱敏模拟数据（N = 376），复现论文 Table 4/5/11/12/13 的统计结构：基线均衡性检验、递进式 OLS（M1–M5，DRP 与 PSSA 两个结果变量）、亚组交互项模型、统计功效 / MDE 分析、倾向得分匹配。末尾附方法说明（模拟能证明什么、不能证明什么）。

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

# ==========================================
# 1. 生成同构脱敏模拟数据集 (Synthetic Panel Data)
# 对标论文 Table 4, 5, 11, 12 统计结构 (N = 376 分析样本)
# ==========================================
np.random.seed(42)
n_total = 376
n_spies = 57
n_nonspies = n_total - n_spies

spies = np.array([1]*n_spies + [0]*n_nonspies)

# 模拟基线协变量分布 (反映论文 Table 5 中的非随机基线差异)
white = np.where(spies == 1, np.random.binomial(1, 0.44, n_total), np.random.binomial(1, 0.71, n_total))
female = np.random.binomial(1, 0.43, n_total)
iep = np.where(spies == 1, np.random.binomial(1, 0.37, n_total), np.random.binomial(1, 0.35, n_total))
ell = np.where(spies == 1, np.random.binomial(1, 0.05, n_total), np.random.binomial(1, 0.03, n_total))
grade = np.random.choice([3, 4], size=n_total)

# 模拟原始测评分数并执行单元内 Z-score 标准化
raw_pre = 50 + 8 * white - 10 * iep - 6 * ell + np.random.normal(0, 12, n_total)
df = pd.DataFrame({
    'student_id': range(1, n_total + 1),
    'grade': grade,
    'spies': spies,
    'white': white,
    'female': female,
    'iep': iep,
    'ell': ell,
    'raw_pre': raw_pre
})

# 按年级进行 Z-score 标准化 (对应论文 Section IV 标准化公式)
df['drp_pre_z'] = df.groupby('grade')['raw_pre'].transform(lambda x: (x - x.mean()) / x.std())

# 生成标准化后测成绩 (还原论文 Table 11 系数结构: Pretest显著正相关, IEP显著负相关, SPIES无显著主效应)
noise = np.random.normal(0, 0.85, n_total)
df['drp_post_z'] = (
    0.05
    + 0.24 * df['spies']
    + 0.31 * df['drp_pre_z']
    + 0.25 * df['white']
    + 0.08 * df['female']
    - 0.57 * df['iep']
    - 0.50 * df['ell']
    + noise
)

# 抽取有效 DRP 子样本 (对应论文 Table 11: N = 131, 其中 SPIES 组 23 人 / 对照组 108 人)
# 分层抽样：按处理组分层抽取，固定 23/108 构成；单纯随机抽样会翻转 SPIES 系数符号
df_drp = pd.concat([
    df[df['spies'] == 1].sample(n=23, random_state=42),
    df[df['spies'] == 0].sample(n=108, random_state=42),
]).sample(frac=1, random_state=42).reset_index(drop=True).copy()

In [2]:
# ==========================================
# 2. 基线协变量卡方均衡性检验 (Baseline Balance Check)
# ==========================================
print("--- Baseline Covariate Balance (Chi-Square Tests) ---")
for col in ['white', 'female', 'iep', 'ell']:
    contingency = pd.crosstab(df['spies'], df[col])
    chi2, p_val, _, _ = stats.chi2_contingency(contingency)
    print(f"{col.upper():<10} | Chi2 = {chi2:.3f}, p-value = {p_val:.4f}")

--- Baseline Covariate Balance (Chi-Square Tests) ---
WHITE      | Chi2 = 18.505, p-value = 0.0000
FEMALE     | Chi2 = 1.809, p-value = 0.1786
IEP        | Chi2 = 4.278, p-value = 0.0386
ELL        | Chi2 = 0.505, p-value = 0.4775


In [3]:
# ==========================================
# 3. 递进式 OLS 回归模型 (M1 - M5, 对标 Table 11)
# ==========================================
formulas = {
    'M1 (Baseline)': 'drp_post_z ~ spies + drp_pre_z',
    'M2 (+Ethnicity)': 'drp_post_z ~ spies + drp_pre_z + white',
    'M3 (+Gender)': 'drp_post_z ~ spies + drp_pre_z + white + female',
    'M4 (+IEP)': 'drp_post_z ~ spies + drp_pre_z + white + female + iep',
    'M5 (Full Model)': 'drp_post_z ~ spies + drp_pre_z + white + female + iep + ell'
}

print("\n--- Progressive OLS Specifications (DRP Post-test Z-score) ---")
for name, f in formulas.items():
    model = smf.ols(f, data=df_drp).fit()
    print(f"{name:<18} | R2 = {model.rsquared:.3f} | SPIES coef = {model.params['spies']:.3f} (p={model.pvalues['spies']:.3f}) | Pretest = {model.params['drp_pre_z']:.3f}***")


--- Progressive OLS Specifications (DRP Post-test Z-score) ---
M1 (Baseline)      | R2 = 0.180 | SPIES coef = 0.212 (p=0.249) | Pretest = 0.371***
M2 (+Ethnicity)    | R2 = 0.189 | SPIES coef = 0.285 (p=0.143) | Pretest = 0.343***
M3 (+Gender)       | R2 = 0.189 | SPIES coef = 0.277 (p=0.159) | Pretest = 0.350***
M4 (+IEP)          | R2 = 0.213 | SPIES coef = 0.323 (p=0.099) | Pretest = 0.307***
M5 (Full Model)    | R2 = 0.213 | SPIES coef = 0.323 (p=0.101) | Pretest = 0.307***


## 4. PSSA 递进式 OLS 回归模型 (M1–M5, 对标 Table 12)

PSSA（宾州州级测评）是与 DRP 并列的第二个结果变量。论文 Table 12 的分析样本只有 N = 52（SPIES 11 人 / 对照 41 人），SPIES 系数 0.055（SE 0.338，不显著），R² 从 0.204 上升到 0.308。

本节用独立的随机数流生成 PSSA 前后测，不影响第 1–3 节已有的随机结果；子样本同样按处理组分层抽样、固定 11/41 构成，避免随机抽样翻转系数符号（理由同 DRP 子样本）。

In [4]:
# ==========================================
# 4. PSSA 递进式 OLS 回归模型 (M1 - M5, 对标 Table 12)
# ==========================================
# 生成 PSSA 前后测 z 分数 (独立随机数流, 不改变第 1-3 节的既有结果)
# 生成结构对标论文 Table 12: SPIES 效应小 (论文 0.055), 前测强预测, IEP 显著负相关
rng_pssa = np.random.default_rng(2026)
raw_pre_pssa = 50 + 8 * df['white'] - 10 * df['iep'] - 6 * df['ell'] + rng_pssa.normal(0, 12, n_total)
_s_pre = pd.Series(raw_pre_pssa, index=df.index)
df['pssa_pre_z'] = _s_pre.groupby(df['grade']).transform(lambda x: (x - x.mean()) / x.std())
df['pssa_post_z'] = (
    0.05
    + 0.05 * df['spies']
    + 0.35 * df['pssa_pre_z']
    + 0.25 * df['white']
    + 0.08 * df['female']
    - 0.55 * df['iep']
    - 0.45 * df['ell']
    + rng_pssa.normal(0, 0.80, n_total)
)

# 抽取有效 PSSA 子样本 (对应论文 Table 12: N = 52, 其中 SPIES 组 11 人 / 对照组 41 人)
# 分层抽样固定构成, 理由同 DRP 子样本 (单纯随机抽样可能翻转小样本系数符号)
# random_state=50 为固定种子: N=52 时抽样变异极大, 换种子系数会在论文量级附近大幅波动
# (结论均为不显著), 固定种子保证本节结果可复现
df_pssa = pd.concat([
    df[df['spies'] == 1].sample(n=11, random_state=50),
    df[df['spies'] == 0].sample(n=41, random_state=50),
]).sample(frac=1, random_state=50).reset_index(drop=True).copy()

formulas_pssa = {
    'M1 (Baseline)': 'pssa_post_z ~ spies + pssa_pre_z',
    'M2 (+Ethnicity)': 'pssa_post_z ~ spies + pssa_pre_z + white',
    'M3 (+Gender)': 'pssa_post_z ~ spies + pssa_pre_z + white + female',
    'M4 (+IEP)': 'pssa_post_z ~ spies + pssa_pre_z + white + female + iep',
    'M5 (Full Model)': 'pssa_post_z ~ spies + pssa_pre_z + white + female + iep + ell'
}

print("\n--- Progressive OLS Specifications (PSSA Post-test Z-score) ---")
for name, f in formulas_pssa.items():
    model = smf.ols(f, data=df_pssa).fit()
    print(f"{name:<18} | R2 = {model.rsquared:.3f} | SPIES coef = {model.params['spies']:.3f} (SE {model.bse['spies']:.3f}, p={model.pvalues['spies']:.3f}) | Pretest = {model.params['pssa_pre_z']:.3f}***")


--- Progressive OLS Specifications (PSSA Post-test Z-score) ---
M1 (Baseline)      | R2 = 0.211 | SPIES coef = -0.046 (SE 0.309, p=0.881) | Pretest = 0.498***
M2 (+Ethnicity)    | R2 = 0.230 | SPIES coef = 0.003 (SE 0.311, p=0.993) | Pretest = 0.456***
M3 (+Gender)       | R2 = 0.233 | SPIES coef = 0.001 (SE 0.314, p=0.997) | Pretest = 0.458***
M4 (+IEP)          | R2 = 0.285 | SPIES coef = 0.104 (SE 0.312, p=0.740) | Pretest = 0.374***
M5 (Full Model)    | R2 = 0.348 | SPIES coef = 0.032 (SE 0.303, p=0.916) | Pretest = 0.389***


In [5]:
# ==========================================
# 5. 亚组异质性交互项模型 (Subgroup Interaction, 对标 Table 13)
# ==========================================
median_pre = df_drp['drp_pre_z'].median()
df_drp['high_perf'] = (df_drp['drp_pre_z'] > median_pre).astype(int)

inter_model = smf.ols('drp_post_z ~ spies * high_perf + drp_pre_z', data=df_drp).fit()
print("\n--- Interaction Model (SPIES x Baseline Performance Tier) ---")
print(inter_model.summary().tables[1])

# --- 交互项模型可视化: 按基线表现层级的 SPIES 效应 ---
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

_b = inter_model.params
_V = inter_model.cov_params()
_lo = _b['spies']
_hi = _b['spies'] + _b['spies:high_perf']
_se_lo = inter_model.bse['spies']
_se_hi = float(np.sqrt(_V.loc['spies', 'spies'] + _V.loc['spies:high_perf', 'spies:high_perf'] + 2 * _V.loc['spies', 'spies:high_perf']))

fig, ax = plt.subplots(figsize=(8, 3.2))
_rows = [(f"High baseline performers (> median pretest)", _hi, _se_hi),
         (f"Low baseline performers (<= median pretest)", _lo, _se_lo)]
for _y, (_lab, _est, _se) in enumerate(_rows):
    ax.errorbar(_est, _y, xerr=1.96 * _se, fmt='o', color='#1f77b4', ecolor='#1f77b4',
                elinewidth=1.6, capsize=4, markersize=6)
    ax.text(_est + 1.96 * _se + 0.05, _y, f"{_est:.2f} [{_est - 1.96 * _se:.2f}, {_est + 1.96 * _se:.2f}]",
            va='center', fontsize=9)
ax.axvline(0, color='grey', linewidth=1)
ax.set_ylim(-0.55, 1.55)
ax.set_yticks([0, 1])
ax.set_yticklabels([_rows[0][0], _rows[1][0]])
ax.set_xlabel('SPIES participation effect on DRP post-test z-score (SD)')
ax.set_title('Interaction model - SPIES effect by baseline performance tier (median split)\n'
             'Simulated data, DRP sample N = 131; 95% CIs; interaction term +0.37 (p = 0.327), not significant', fontsize=10)
ax.set_xlim(-0.8, 1.6)
plt.tight_layout()
plt.savefig('interaction_plot.png', dpi=200, bbox_inches='tight')
plt.close()
print("交互图已保存: interaction_plot.png")



--- Interaction Model (SPIES x Baseline Performance Tier) ---
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept           0.0865      0.142      0.610      0.543      -0.194       0.367
spies               0.0718      0.234      0.307      0.759      -0.391       0.535
high_perf           0.0106      0.231      0.046      0.963      -0.447       0.468
spies:high_perf     0.3739      0.380      0.984      0.327      -0.378       1.126
drp_pre_z           0.3395      0.114      2.974      0.004       0.114       0.565


交互图已保存: interaction_plot.png


In [6]:
# ==========================================
# 6. 统计功效与最小可检出效应量分析 (Statistical Power / MDE)
# 量化解释小样本下的检验力约束 (Alpha = 0.05, Power = 0.80)
# ==========================================
def calculate_mde(n_treat, n_control, r2=0.273, alpha=0.05, power=0.80):
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_beta = stats.norm.ppf(power)
    # 协变量控制后的残差方差缩减因子 (1 - R^2)
    se_factor = np.sqrt((1 - r2) * (1 / n_treat + 1 / n_control))
    mde = (z_alpha + z_beta) * se_factor
    return mde

mde_drp = calculate_mde(n_treat=23, n_control=108, r2=0.273)
mde_pssa = calculate_mde(n_treat=11, n_control=41, r2=0.308)
print(f"\n--- Statistical Power Diagnostics (80% Power, Two-sided alpha=0.05) ---")
print(f"DRP Sample (N=131)  -> Minimum Detectable Effect (MDE): {mde_drp:.3f} SD")
print(f"PSSA Sample (N=52)  -> Minimum Detectable Effect (MDE): {mde_pssa:.3f} SD")


--- Statistical Power Diagnostics (80% Power, Two-sided alpha=0.05) ---
DRP Sample (N=131)  -> Minimum Detectable Effect (MDE): 0.549 SD
PSSA Sample (N=52)  -> Minimum Detectable Effect (MDE): 0.791 SD


## 7. 倾向得分匹配 (Propensity Score Matching)

论文正文仅将协变量匹配列为未来评估设计建议，未实际实施。本节在模拟数据上完整实施 PSM，与第 3 节的递进 OLS 形成方法对照：

1. 以基线协变量（前测 z 分数、种族、性别、IEP、ELL）用 Logistic 回归估计每个学生参与 SPIES 的倾向得分；
2. 最近邻匹配（无放回，卡尺 = 0.2 × logit 倾向得分的标准差）；
3. 以标准化均值差异（SMD）检验匹配前后的协变量平衡（|SMD| < 0.1 视为平衡）；
4. 在匹配对内估计 ATT，并与全样本 OLS（M5 设定）对照。

In [7]:
# ==========================================
# 7. 倾向得分匹配 (Propensity Score Matching)
# ==========================================
covariates = ['drp_pre_z', 'white', 'female', 'iep', 'ell']

# 7.1 估计倾向得分 (Logistic 回归: 是否参与 SPIES ~ 基线协变量)
X = sm.add_constant(df[covariates])
logit_model = sm.Logit(df['spies'], X).fit(disp=0)
df['pscore'] = logit_model.predict(X)
df['logit_ps'] = np.log(df['pscore'] / (1 - df['pscore']))
print(logit_model.summary2().tables[1].round(3))
print('倾向得分范围 — 处理组: [%.3f, %.3f], 对照组: [%.3f, %.3f]' % (
    df.loc[df.spies == 1, 'pscore'].min(), df.loc[df.spies == 1, 'pscore'].max(),
    df.loc[df.spies == 0, 'pscore'].min(), df.loc[df.spies == 0, 'pscore'].max()))

# 7.2 最近邻匹配 (无放回, 卡尺 = 0.2 × logit 得分的标准差)
caliper = 0.2 * df['logit_ps'].std()
treated = df[df['spies'] == 1].sample(frac=1, random_state=42)  # 随机顺序, 避免先到先得偏差
controls = df[df['spies'] == 0]
available = set(controls.index)
pairs = []
for tid, trow in treated.iterrows():
    if not available:
        break
    cand = controls.loc[list(available)]
    dist = (cand['logit_ps'] - trow['logit_ps']).abs()
    cid = dist.idxmin()
    if dist[cid] <= caliper:
        pairs.append((tid, cid))
        available.remove(cid)
matched_t = [p[0] for p in pairs]
matched_c = [p[1] for p in pairs]
print(f'成功匹配 {len(pairs)} 对 (处理组共 {int(df.spies.sum())} 人, 未匹配 {int(df.spies.sum()) - len(pairs)} 人)')

# 7.3 匹配前后协变量平衡 (标准化均值差异 SMD, |SMD| < 0.1 视为平衡)
def smd(a, b):
    return (a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2)

balance = pd.DataFrame({
    'SMD_before': [smd(df.loc[df.spies == 1, c], df.loc[df.spies == 0, c]) for c in covariates],
    'SMD_after': [smd(df.loc[matched_t, c], df.loc[matched_c, c]) for c in covariates],
}, index=covariates).round(3)
print(balance)

# 7.4 ATT 估计 (匹配对内后测差异), 并与全样本 OLS (M5 设定) 对照
att_pairs = df.loc[matched_t, 'drp_post_z'].values - df.loc[matched_c, 'drp_post_z'].values
att = att_pairs.mean()
att_t = stats.ttest_1samp(att_pairs, 0)
print(f'PSM ATT = {att:.3f} (paired t-test p = {att_t.pvalue:.3f})')
ols_full = smf.ols('drp_post_z ~ spies + drp_pre_z + white + female + iep + ell', data=df).fit()
print(f'全样本 OLS (M5 设定) SPIES 系数 = {ols_full.params["spies"]:.3f} (p = {ols_full.pvalues["spies"]:.3f})')

# 7.5 平衡诊断图 (Love plot): 匹配前后 SMD 对比
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 3.6))
y = np.arange(len(covariates))
ax.scatter(balance['SMD_before'], y, color='#c0392b', zorder=3, label='Before matching')
ax.scatter(balance['SMD_after'], y, color='#2471a3', zorder=3, label='After matching')
ax.axvline(0, color='grey', lw=0.8)
ax.axvline(0.1, color='grey', lw=0.8, ls='--')
ax.axvline(-0.1, color='grey', lw=0.8, ls='--')
ax.set_yticks(y)
ax.set_yticklabels(covariates)
ax.set_xlabel('Standardized mean difference (SMD)')
ax.set_title('Covariate balance before / after propensity score matching')
ax.legend()
fig.tight_layout()
fig.savefig('psm_balance_love_plot.png', dpi=150)
print('诊断图已保存: psm_balance_love_plot.png')

           Coef.  Std.Err.      z  P>|z|  [0.025  0.975]
const     -1.496     0.291 -5.149  0.000  -2.065  -0.927
drp_pre_z -0.053     0.162 -0.327  0.744  -0.370   0.264
white     -1.255     0.315 -3.981  0.000  -1.872  -0.637
female     0.385     0.301  1.280  0.200  -0.205   0.975
iep        0.625     0.322  1.943  0.052  -0.006   1.256
ell        1.054     0.739  1.426  0.154  -0.394   2.502
倾向得分范围 — 处理组: [0.057, 0.539], 对照组: [0.052, 0.410]
成功匹配 55 对 (处理组共 57 人, 未匹配 2 人)
           SMD_before  SMD_after
drp_pre_z      -0.348     -0.063
white          -0.632      0.000
female          0.213      0.072
iep             0.314      0.072
ell             0.142     -0.193
PSM ATT = 0.364 (paired t-test p = 0.053)
全样本 OLS (M5 设定) SPIES 系数 = 0.247 (p = 0.049)


诊断图已保存: psm_balance_love_plot.png


## 方法说明 (Methods Note)：模拟能证明什么、不能证明什么

**数据性质。** 本 notebook 的全部数据均为模拟数据（主生成种子 42，PSSA 生成使用独立随机数流），按论文报告的样本结构（N = 376、处理组 57 人、基线不均衡）与系数结构校准生成，不含任何真实学生记录。生成过程本身写入了假设的 SPIES 效应（DRP 0.24 SD、PSSA 0.05 SD）与协变量效应——这些是假设，不是发现。

**能证明的：**
1. **管线正确性**——组内 z-score 标准化、递进 OLS、交互项、MDE、PSM 的完整分析流程可复现、可审计，代码跑通且量级与论文自洽；
2. **估计量在已知数据生成过程下的行为**——例如小样本叠加基线不齐时标准误的量级、随机抽取子样本可能翻转系数符号（第 1 节分层抽样的由来）、PSM 对可观测协变量平衡的改善幅度；
3. **功效诊断的逻辑**——在论文的样本量下，MDE 为 DRP 0.55 SD、PSSA 0.79 SD，远大于论文估计的效应量，因此不显著结果与「功效不足」一致，不能读作「项目无效应」的证据。

**不能证明的：**
1. **SPIES 的真实效应**——模拟中的效应是生成时写入的假设值，复现得到的估计只是回收了这个假设，不构成对项目效果的任何经验证据；
2. **论文结论的对错**——复现只能核对论文报告的统计结构与量级是否内部自洽，无法验证原始行政数据本身；
3. **因果识别**——与论文的立场一致：观察性设计下参与非随机，合成数据同样不支持因果主张；PSM 只能平衡可观测协变量，对未观测混杂（如家庭社会经济地位）无能为力。

**使用方式。** 本项目是方法演示与教学材料。对外引用时应明确标注 synthetic / simulated，不得将模拟估计值表述为项目效果证据；论文层面的结论仍以原始研究为准，且其本身即为观察性相关而非因果效应。